# 01. First look at one BraTS 2021 case

**What this notebook is for.** It looks inside the Kaggle-hosted copy of the BraTS 2021 training data, counts the cases, and opens one case to see what it is made of: four MRI scans and one label map (the expert-approved reference outline).

**How it fits the project.** This is Session 3 of Brain Tumor Viewer, a research and education project by Debraj Karmaker. **Not for diagnosis.** Nothing here trains or runs a model. The facts it prints are recorded in `docs/DATA_ACCESS.md`, section 8.4.

**Where it ran.** On Kaggle, not on a laptop, on Nov 16, 2026, with the dataset `dschettler8845/brats-2021-task1` attached (Kaggle dataset ID 1541666, version 1). No GPU and no internet access. Python 3.13, nibabel 5.4.2.

**What is real and what is missing.** The text printed under each cell is the real output from that run, exactly as Kaggle saved it. One thing was removed on purpose: the picture drawn by Cell 6, which shows a slice of a real scan. Scan images are never committed to this repository (see `CLAUDE.md` and `docs/DATA_ACCESS.md`). Run the notebook on Kaggle to see the picture.

**Run the cells in order.** Each cell uses things the earlier cells set up. If the Kaggle session restarts, its memory is wiped and the cells must be run again from the top. Cells 3 and 4 each read through a 13 GB archive and take a few minutes.

In [2]:
# ===== CELL 1: what did Kaggle attach? =====
# Kaggle puts every attached dataset under the folder /kaggle/input.
# os.walk visits every folder inside it, like clicking through folders by hand.
# For each file we print where it is and how big it is, and we remember where
# it is in "paths" (a lookup table: file name -> full location).
import os

INPUT_FOLDER = "/kaggle/input"
paths = {}

for folder, subfolders, files in os.walk(INPUT_FOLDER):
    for name in files:
        location = os.path.join(folder, name)
        paths[name] = location
        print(location, "|", f"{os.path.getsize(location):,}", "bytes")

/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_00495.tar | 10,321,920 bytes
/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_00621.tar | 10,772,480 bytes
/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar | 13,396,408,320 bytes


In [3]:
# ===== CELL 2: look inside the small archive (one case) =====
# A .tar file is many files packed into one, like a taped-up box.
# tarfile can read the packing list without unpacking anything.
import tarfile

small_archive = tarfile.open(paths["BraTS2021_00495.tar"])
for item in small_archive.getmembers():
    if item.isfile():
        print(item.name, "|", f"{item.size:,}", "bytes")
small_archive.close()

./BraTS2021_00495_flair.nii.gz | 2,503,035 bytes
./BraTS2021_00495_seg.nii.gz | 37,663 bytes
./BraTS2021_00495_t1.nii.gz | 2,652,585 bytes
./BraTS2021_00495_t1ce.nii.gz | 2,689,385 bytes
./BraTS2021_00495_t2.nii.gz | 2,428,304 bytes


In [4]:
# ===== CELL 3: read the packing list of the big archive =====
# This reads only the labels on the box, not the scans, so it needs no GPU.
# It may take a few minutes because the archive is 13 GB.
big_archive = tarfile.open(paths["BraTS2021_Training_Data.tar"])

case_ids = []          # every case ID we meet, once each
file_count = 0         # how many files in total
odd_files = []         # files that are not inside a folder named after their case
case_00118_files = []  # the case we know TCIA's listed sizes for

for item in big_archive:
    if not item.isfile():
        continue                      # skip folders; count files only
    file_count = file_count + 1
    if file_count <= 5:
        print("Example entry:", item.name)

    # A file name looks like BraTS2021_00118_flair.nii.gz.
    # Its first 15 characters are the case ID: BraTS2021_00118.
    file_name = os.path.basename(item.name)
    case_id = file_name[:15]
    folder_name = os.path.basename(os.path.dirname(item.name))
    if case_id not in case_ids:
        case_ids.append(case_id)
    if folder_name != case_id:
        odd_files.append(item.name)
    if case_id == "BraTS2021_00118":
        case_00118_files.append((file_name, item.size))

big_archive.close()

print()
print("Number of cases:", len(case_ids))
print("Number of files:", file_count)
print("Files per case if even:", file_count / len(case_ids))
print("Lowest and highest case ID:", min(case_ids), max(case_ids))
print("Files not inside their own case folder:", len(odd_files), odd_files[:5])
print("Is case 00495 also in the big archive?", "BraTS2021_00495" in case_ids)
print("Is case 00621 also in the big archive?", "BraTS2021_00621" in case_ids)
print()
print("Case 00118, to compare with the sizes TCIA lists:")
for file_name, size in sorted(case_00118_files):
    print(file_name, "|", f"{size:,}", "bytes")

Example entry: ./.DS_Store
Example entry: ./BraTS2021_00000/BraTS2021_00000_flair.nii.gz
Example entry: ./BraTS2021_00000/BraTS2021_00000_seg.nii.gz
Example entry: ./BraTS2021_00000/BraTS2021_00000_t1.nii.gz
Example entry: ./BraTS2021_00000/BraTS2021_00000_t1ce.nii.gz

Number of cases: 1252
Number of files: 6256
Files per case if even: 4.996805111821086
Lowest and highest case ID: .DS_Store BraTS2021_01666
Files not inside their own case folder: 1 ['./.DS_Store']
Is case 00495 also in the big archive? True
Is case 00621 also in the big archive? True

Case 00118, to compare with the sizes TCIA lists:
BraTS2021_00118_flair.nii.gz | 1,872,411 bytes
BraTS2021_00118_seg.nii.gz | 128,462 bytes
BraTS2021_00118_t1.nii.gz | 1,936,184 bytes
BraTS2021_00118_t1ce.nii.gz | 1,896,946 bytes
BraTS2021_00118_t2.nii.gz | 2,086,035 bytes


In [5]:
# ===== CELL 4: tidy count, and are the two separately packed cases duplicates? =====
# Cell 3 counted the junk file .DS_Store as if it were a case. Count real cases only.
real_case_ids = []
for case_id in case_ids:
    if case_id.startswith("BraTS2021_"):
        real_case_ids.append(case_id)
print("Real cases:", len(real_case_ids))
print("Lowest and highest real case ID:", min(real_case_ids), max(real_case_ids))
print()

# A fingerprint (SHA-256 checksum) is a short code computed from a file's bytes.
# Two files with the same fingerprint are the same file, byte for byte.
import hashlib

two_cases = ["BraTS2021_00495", "BraTS2021_00621"]
separate_copy = {}   # file name -> fingerprint, from the two small archives
big_copy = {}        # file name -> fingerprint, from the big archive

for case_id in two_cases:
    archive = tarfile.open(paths[case_id + ".tar"])
    for item in archive:
        if item.isfile():
            file_bytes = archive.extractfile(item).read()
            separate_copy[os.path.basename(item.name)] = hashlib.sha256(file_bytes).hexdigest()
    archive.close()

archive = tarfile.open(paths["BraTS2021_Training_Data.tar"])
for item in archive:
    file_name = os.path.basename(item.name)
    if item.isfile() and file_name[:15] in two_cases:
        file_bytes = archive.extractfile(item).read()
        big_copy[file_name] = hashlib.sha256(file_bytes).hexdigest()
archive.close()

for file_name in sorted(set(separate_copy) | set(big_copy)):
    a = separate_copy.get(file_name, "MISSING")
    b = big_copy.get(file_name, "MISSING")
    verdict = "SAME" if a == b else "DIFFERENT"
    print(file_name, "|", verdict, "| separate:", a[:16], "| big:", b[:16])

Real cases: 1251
Lowest and highest real case ID: BraTS2021_00000 BraTS2021_01666

BraTS2021_00495_flair.nii.gz | SAME | separate: c48e10e37713b8bd | big: c48e10e37713b8bd
BraTS2021_00495_seg.nii.gz | SAME | separate: ab96bbc494711d20 | big: ab96bbc494711d20
BraTS2021_00495_t1.nii.gz | SAME | separate: d5f4fbe19a2fd0d6 | big: d5f4fbe19a2fd0d6
BraTS2021_00495_t1ce.nii.gz | SAME | separate: 5b29c167b03c6d1f | big: 5b29c167b03c6d1f
BraTS2021_00495_t2.nii.gz | SAME | separate: 8ff602f806a00cfa | big: 8ff602f806a00cfa
BraTS2021_00621_flair.nii.gz | SAME | separate: 0bde2d4e7e99a87f | big: 0bde2d4e7e99a87f
BraTS2021_00621_seg.nii.gz | SAME | separate: d444fdd972c498d7 | big: d444fdd972c498d7
BraTS2021_00621_t1.nii.gz | SAME | separate: d36c8087f50d26de | big: d36c8087f50d26de
BraTS2021_00621_t1ce.nii.gz | SAME | separate: 6868ed12c7b0d440 | big: 6868ed12c7b0d440
BraTS2021_00621_t2.nii.gz | SAME | separate: 6d038a25334428f9 | big: 6d038a25334428f9


In [6]:
# ===== CELL 5: take one case out of its archive and describe one scan =====
# We unpack the small one-case archive into /kaggle/temp. That folder is a
# scratch area: it is wiped when the session ends and is never saved or published.
import nibabel   # the library that reads .nii.gz scan files
print("nibabel version:", nibabel.__version__)

CASE_ID = "BraTS2021_00495"
CASE_FOLDER = "/kaggle/temp/" + CASE_ID
archive = tarfile.open(paths[CASE_ID + ".tar"])
archive.extractall(CASE_FOLDER, filter="data")   # filter="data" is a safety setting: plain files only
archive.close()
print(sorted(os.listdir(CASE_FOLDER)))

# Open the T1 scan with contrast and describe it in numbers. Nothing is drawn yet.
scan = nibabel.load(CASE_FOLDER + "/" + CASE_ID + "_t1ce.nii.gz")
print("Shape (how many voxels along each of the 3 directions):", scan.shape)
print("Voxel size in millimetres:", scan.header.get_zooms())
print("Number type stored in the file:", scan.get_data_dtype())
print("Orientation codes (the direction each axis counts towards):", nibabel.aff2axcodes(scan.affine))

nibabel version: 5.4.2
['BraTS2021_00495_flair.nii.gz', 'BraTS2021_00495_seg.nii.gz', 'BraTS2021_00495_t1.nii.gz', 'BraTS2021_00495_t1ce.nii.gz', 'BraTS2021_00495_t2.nii.gz']
Shape (how many voxels along each of the 3 directions): (240, 240, 155)
Voxel size in millimetres: (np.float32(1.0), np.float32(1.0), np.float32(1.0))
Number type stored in the file: int16
Orientation codes (the direction each axis counts towards): ('L', 'P', 'S')


In [8]:
# ===== CELL 6: load all five files, list the label values, draw one slice of each =====
import numpy as np                 # the library for grids of numbers
import matplotlib.pyplot as plt    # the library for drawing

kinds = ["t1", "t1ce", "t2", "flair", "seg"]
volumes = {}   # lookup table: kind of file -> its 3D grid of numbers
for kind in kinds:
    image = nibabel.load(CASE_FOLDER + "/" + CASE_ID + "_" + kind + ".nii.gz")
    volumes[kind] = np.asanyarray(image.dataobj)
    print(kind, "| shape", volumes[kind].shape, "| number type", volumes[kind].dtype,
          "| smallest", volumes[kind].min(), "| largest", volumes[kind].max(),
          "| orientation", nibabel.aff2axcodes(image.affine))
print()

# Which code numbers appear in the label map, and how many voxels carry each one?
# We expect 0, 1, 2 and 4. A 3 would mean this is not the 2021 release.
values, counts = np.unique(volumes["seg"], return_counts=True)
for value, count in zip(values, counts):
    print("Label", value, "|", f"{count:,}", "voxels")
print()

# Choose the horizontal slice with the most labelled voxels, so the tumor is in view.
labelled_per_slice = (volumes["seg"] > 0).sum(axis=(0, 1))
slice_number = int(labelled_per_slice.argmax())
print("Showing slice", slice_number, "of", volumes["seg"].shape[2], "(counting from the bottom of the head)")

# Draw the same slice from each file, side by side.
# The second axis counts towards the back of the head (the P in L, P, S), so
# origin="upper" puts the front of the head at the top, the usual way to show a slice.
# This is a quick look only. It does NOT settle which side is left and which is right;
# that is proved separately against 3D Slicer later in the project.
figure, panels = plt.subplots(1, 5, figsize=(20, 4.8))
for panel, kind in zip(panels, kinds):
    picture = volumes[kind][:, :, slice_number].T
    if kind == "seg":
        panel.imshow(picture, cmap="nipy_spectral", vmin=0, vmax=4, origin="upper", interpolation="nearest")
    else:
        panel.imshow(picture, cmap="gray", origin="upper")
    panel.set_title(kind)
    panel.axis("off")
figure.suptitle(CASE_ID + ", slice " + str(slice_number) + ". Research and education only. Not for diagnosis. Left/right not verified.")
plt.show()

t1 | shape (240, 240, 155) | number type int16 | smallest 0 | largest 6402 | orientation ('L', 'P', 'S')
t1ce | shape (240, 240, 155) | number type int16 | smallest 0 | largest 12299 | orientation ('L', 'P', 'S')
t2 | shape (240, 240, 155) | number type int16 | smallest 0 | largest 2508 | orientation ('L', 'P', 'S')
flair | shape (240, 240, 155) | number type int16 | smallest 0 | largest 2289 | orientation ('L', 'P', 'S')
seg | shape (240, 240, 155) | number type uint16 | smallest 0 | largest 4 | orientation ('L', 'P', 'S')

Label 0 | 8,828,698 voxels
Label 1 | 20,808 voxels
Label 2 | 46,447 voxels
Label 4 | 32,047 voxels

Showing slice 84 of 155 (counting from the bottom of the head)
